# Fiche 2 — Joindre deux tables sans perdre de lignes en silence

| | |
|---|---|
| 🎯 **Objectif** | Choisir le type de jointure, compter ce qui se perd et retrouver les clés mal orthographiées avec une anti-jointure |
| ⏱️ **Durée** | 40 min |
| 🧰 **Prérequis** | Fiche 1 |
| 🔗 **Dans le projet** | Étape 1 (fusion des sources) |

> **L'idée clé.** Une jointure compare des **chaînes de caractères exactes** : « United States » et « United States of America » ne se rencontrent jamais. Après chaque jointure, on compte les lignes et on regarde **qui a disparu**.

## 1. Deux sources, deux façons d'écrire les pays

In [1]:
import pandas as pd

from agrireco import io

_, fao_dir = io.download()  # téléchargement vérifié par empreinte SHA-256 (rien à faire s'il est déjà là)

rendements = pd.read_csv(fao_dir / "yield.csv")[["Area", "Item", "Year", "Value"]]
temp = pd.read_csv(fao_dir / "temp.csv").groupby(["country", "year"], as_index=False)["avg_temp"].mean()
temp = temp.rename(columns={"country": "Area", "year": "Year"})
ble_2005 = rendements[(rendements["Item"] == "Wheat") & (rendements["Year"] == 2005)]
print(len(ble_2005), "pays produisent du blé en 2005 dans le fichier FAO")

125 pays produisent du blé en 2005 dans le fichier FAO


In [2]:
jointe = ble_2005.merge(temp, on=["Area", "Year"], how="inner")
print("après jointure interne :", len(jointe), "pays")

après jointure interne : 89 pays


La jointure **interne** (`how="inner"`) ne garde que les lignes présentes des deux côtés. Des pays ont disparu ;
lesquels ? On utilise une jointure **gauche** avec l'indicateur `_merge` : c'est une *anti-jointure*.

In [3]:
gauche = ble_2005.merge(temp, on=["Area", "Year"], how="left", indicator=True)
perdus = gauche.loc[gauche["_merge"] == "left_only", "Area"]
print(len(perdus), "pays perdus, dont :", sorted(perdus)[:12])

36 pays perdus, dont : ['Bhutan', 'Bolivia (Plurinational State of)', 'Bosnia and Herzegovina', 'Chad', 'China, Taiwan Province of', 'China, mainland', 'Cyprus', 'Czechia', "Democratic People's Republic of Korea", 'Democratic Republic of the Congo', 'Eswatini', 'Ethiopia']


## ⚠️ Piège : les plus gros producteurs disparaissent sans message d'erreur

In [4]:
for pays in ["United States of America", "Russian Federation", "Iran (Islamic Republic of)"]:
    print(f"{pays:<27} dans le fichier FAO : {pays in set(ble_2005['Area'])} | après jointure : {pays in set(jointe['Area'])}")

United States of America    dans le fichier FAO : True | après jointure : False
Russian Federation          dans le fichier FAO : True | après jointure : False
Iran (Islamic Republic of)  dans le fichier FAO : True | après jointure : False


Les États-Unis et la Russie, deux des plus gros producteurs de blé au monde, sont sortis de l'analyse : pandas n'a
rien signalé, parce qu'une jointure qui ne trouve pas de correspondance n'est pas une erreur. Il faut **compter**.

## 2. Retrouver les noms qui ne se correspondent pas
On liste les noms présents d'un côté et absents de l'autre, puis on cherche à la main les correspondances.

In [5]:
noms_fao = set(rendements["Area"])
noms_temp = set(temp["Area"])
absents = sorted(noms_temp - noms_fao)
print(len(absents), "noms de temp.csv absents de la FAO, par exemple :", absents[:8])

19 noms de temp.csv absents de la FAO, par exemple : ['Bolivia', 'Bosnia And Herzegovina', 'Congo (Democratic Republic Of The)', 'Czech Republic', "Côte D'Ivoire", 'Guinea Bissau', 'Hong Kong', 'Iran']


Repère `Bosnia And Herzegovina` : seule une **majuscule** diffère de `Bosnia and Herzegovina`. Les autres noms
sont à retrouver toi-même dans le projet.

In [6]:
# On commence une table de correspondance (à compléter et à tester dans le projet)
correspondance = {"United States": "United States of America", "Russia": "Russian Federation"}
temp_corrige = temp.assign(Area=temp["Area"].replace(correspondance))
jointe2 = ble_2005.merge(temp_corrige, on=["Area", "Year"], how="inner")
print("avant :", len(jointe), "pays | après 2 corrections :", len(jointe2), "pays")

avant : 89 pays | après 2 corrections : 91 pays


### ✍️ À toi de jouer

Le fichier de pluie (`rainfall.csv`) a lui aussi ses propres noms (et une colonne ` Area` avec une espace au début !). Liste les noms de `rainfall.csv` absents du fichier FAO et trouve à quel nom FAO correspondent 3 d'entre eux.

In [7]:
# Ta réponse ici

<details>
<summary><b>💡 Voir une solution</b> (après avoir cherché !)</summary>

```python
pluie = pd.read_csv(fao_dir / "rainfall.csv")
pluie.columns = pluie.columns.str.strip()      # « ␣Area » → « Area »
absents = sorted(set(pluie["Area"].str.strip()) - noms_fao)
print(absents)
# Exemples : 'Vietnam' → 'Viet Nam', 'Iran' → 'Iran (Islamic Republic of)', 'Venezuela, RB' → 'Venezuela (Bolivarian Republic of)'
print([n for n in noms_fao if "Viet" in n or "Iran" in n or "Venezuela" in n])
```

**Pourquoi ?** chaque organisme a sa convention (Banque mondiale, FAO, Berkeley Earth…). Une table de correspondance **explicite**, versionnée et testée, vaut mieux qu'un rapprochement « flou » automatique qui pourrait marier deux pays différents (Niger et Nigeria !). Note aussi les espaces parasites : `str.strip()` avant de comparer.
</details>

## ✅ À retenir

1. Après chaque jointure : nombre de lignes avant / après, et liste de ce qui a disparu.
2. `merge(..., how='left', indicator=True)` puis `_merge == 'left_only'` : l'anti-jointure qui montre les pertes.
3. Une clé qui ne correspond pas n'est pas une erreur pour pandas : c'est une perte silencieuse.
4. On corrige avec une table de correspondance explicite, documentée et testée.
5. On nettoie les clés (espaces, majuscules) avant de comparer.